Assignment

1. Load prices and form momentum features
2. Generate 5 dummy macro series whose hourly changes predict subsequent returns (non-linear + noise)  
3. Add 3 macro features as linear combinations of those five  
4. Run regression models  
5. Run classification models

In [22]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import (
    AdaBoostClassifier,
    AdaBoostRegressor,
    BaggingClassifier,
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import BayesianRidge, ElasticNet, HuberRegressor, Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import NuSVR, SVR
from sklearn.metrics import precision_score, recall_score, f1_score

%load_ext autoreload
%autoreload 2

sys.path.append("../../")

MARKET_DATA_PATH = "../../market_data/intraday/"
RNG = np.random.default_rng()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
TICKER = "SPY"
PERIOD = 5
DOWNSAMPLING = 60

### Load and downsample prices

In [3]:
# load raw prices and downsample
fp = f"{MARKET_DATA_PATH}{TICKER}_{PERIOD}mn.csv"
prices_raw = pd.read_csv(fp, index_col=0, parse_dates=True)[["Close"]]
prices_raw.columns = [TICKER]
prices_raw.tail()

prices = prices_raw.resample(f"{DOWNSAMPLING}min").last().dropna()
T = len(prices)
idx = prices.index

prices.tail()

,SPY
Date,
2026-03-17 15:00:00+00:00,671.01
2026-03-17 16:00:00+00:00,670.95
2026-03-17 17:00:00+00:00,671.70
2026-03-17 18:00:00+00:00,671.09
2026-03-17 19:00:00+00:00,670.73


In [4]:
# std of returns (used to make features noisy)
sigma = prices.pct_change().std()

### Select features

In [5]:
# momentum: k-period simple log-returns
mom = pd.DataFrame(index=prices.index)
for k in (1, 4, 12, 24):
    mom[f"mom_{k}"] = np.log(prices / prices.shift(k))

In [6]:
# 5 macro indicators predictive of next return
pct_chg = prices.pct_change().shift(-1)    # future shifted prices

macro = pd.DataFrame(index=idx)

# macro 1
innov1 = pd.DataFrame(RNG.normal(0, sigma/2, T), index=idx, columns=[TICKER])
macro[f"macro_1"] = 0.35 * np.sin(pct_chg * np.pi / 24) + innov1    # dummy non-linear function

# macro 2
innov2 = pd.DataFrame(RNG.normal(0, sigma/3, T), index=idx, columns=[TICKER])
macro[f"macro_2"] = 0.05*pct_chg**2 - 0.005*np.log(abs(pct_chg)+0.00001) + innov2

# macro 3
innov3 = pd.DataFrame(RNG.normal(0, sigma, T), index=idx, columns=[TICKER])
macro[f"macro_3"] = 0.00001*pct_chg**3 + innov3

# macro 4
innov4 = pd.DataFrame(RNG.normal(0, sigma*0.4, T), index=idx, columns=[TICKER])
macro[f"macro_4"] = 0.00001*np.tanh(pct_chg*100) + innov4

# macro 5
innov5 = pd.DataFrame(RNG.normal(0, sigma*0.7, T), index=idx, columns=[TICKER])
macro[f"macro_5"] = 5*pct_chg - 0.015*np.cos(pct_chg*100) + innov5

In [7]:
px.line(macro)

In [8]:
# 3 linear combinations of macros (to introduce collinearity)
macro["macro_6"] = macro.eval("0.4 * macro_1 + 0.6 * macro_2")
macro["macro_7"] = macro.eval("0.5 * macro_3 - 0.5 * macro_4 + 0.2 * macro_5")
macro["macro_8"] = macro.eval("0.2 * macro_1 + 0.3 * macro_4 + 0.5 * macro_5")

# X: momentum + 8 dummy macro features
feat = pd.concat([mom, macro], axis=1)

In [9]:
# combine features with target variables
data_regression = pd.concat([feat.shift(1), prices.pct_change()], axis=1).dropna(axis=0)
data_classification = pd.concat([feat.shift(1), np.sign(prices.pct_change())], axis=1).dropna(axis=0)

feature_cols = [c for c in data_regression.columns if c.startswith(("mom_", "macro_"))]

In [10]:
# features + targets for regression
X = data_regression[feature_cols]
y = data_regression[TICKER]

X.shape

(3464, 12)

### Split Data for Cross-Validation

In [11]:
# Train + validation (expanding CV) vs final test holdout
TEST_FRAC = 0.2    # 20% holdout for OOS evaluation
n_test = max(1, int(len(X) * TEST_FRAC))
X_tv, X_test = X.iloc[:-n_test], X.iloc[-n_test:]

# targets for regression models: returns
y_ret_tv, y_ret_test = y.iloc[:-n_test], y.iloc[-n_test:]

# targets for classification models: direction
y_dir_tv, y_dir_test = np.sign(y_ret_tv), np.sign(y_ret_test)

N_SPLITS = 4
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

In [12]:
# check that the splits generate expanding windows
for fold, (tr, va) in enumerate(tscv.split(X_tv)):
    print(f'Fold {fold}: len train (%) {100*len(tr)/len(X_tv):.1f}')


Fold 0: len train (%) 20.1
Fold 1: len train (%) 40.0
Fold 2: len train (%) 60.0
Fold 3: len train (%) 80.0


### Run Regression Models

In [13]:
reg_models = {
    # linear regression models
    "OLS": Pipeline([("scaler", StandardScaler()), ("est", LinearRegression())]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("est", Lasso(alpha=1e-4, max_iter=20_000))]),
    "Ridge":
        Pipeline([("scaler", StandardScaler()), ("est", Ridge(alpha=1.0))]),
    "Huber": Pipeline([("scaler", StandardScaler()), ("est", HuberRegressor(epsilon=1.35, max_iter=200))]),
    "ElasticNet": 
        Pipeline([("scaler", StandardScaler()), ("est", ElasticNet(alpha=1e-3, l1_ratio=0.5, max_iter=10_000))]),
    "BayesianRidge": Pipeline([("scaler", StandardScaler()), ("est", BayesianRidge())]),
    # decision trees (don't require scaling)
    "CART": DecisionTreeRegressor(max_depth=10, random_state=42, min_samples_leaf=15),
    "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "GBRT": GradientBoostingRegressor(random_state=42, max_depth=4, n_estimators=200, learning_rate=0.05),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3),
        n_estimators=150,
        learning_rate=0.05,
        random_state=42,
    ),
    # SVM
    "SVR_RBF": Pipeline([("scaler", StandardScaler()), ("est", SVR(kernel="rbf", C=1.0, epsilon=0.01))]),
    "NuSVR": Pipeline([("scaler", StandardScaler()), ("est", NuSVR(C=1.0, nu=0.5))]),
    # KNN
    "KNN": Pipeline(
        [("scaler", StandardScaler()), ("est", KNeighborsRegressor(n_neighbors=7, weights="distance", n_jobs=-1))]
    ),
    # MLP
    "MLP": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "est",
                MLPRegressor(
                    hidden_layer_sizes=(64, 32),
                    max_iter=800,
                    random_state=42,
                    early_stopping=True,
                ),
            ),
        ]
    ),
}

In [14]:
# calculate cross-validation metrics for a regression model
def cv_regression(estimator, X_, y_):
    rows = []
    for fold, (tr, va) in enumerate(tscv.split(X_)):
        m = clone(estimator)
        m.fit(X_.iloc[tr], y_.iloc[tr])
        pred = m.predict(X_.iloc[va])
        rmse = np.sqrt(mean_squared_error(y_.iloc[va], pred))
        mae = mean_absolute_error(y_.iloc[va], pred)
        r2 = r2_score(y_.iloc[va], pred)
        # actual pnl from trading the prediction
        pnls = np.sign(pred) * y_.iloc[va]
        eq = (1.0 + pnls).cumprod()
        std = pnls.std()
        z = np.mean(pnls) / std
        rows.append({"rmse": rmse, "mae": mae, "r2": r2, "pnl": eq.iloc[-1], "zscore": z})
    return pd.DataFrame(rows)

cv_tables = {k: cv_regression(m, X_tv, y_ret_tv) for k, m in reg_models.items()}
cv_summary = pd.DataFrame({k: v.mean(numeric_only=True) for k, v in cv_tables.items()}).T
# sort models by zscore then rmse
cv_summary = cv_summary.sort_values(by=["zscore", "rmse"], ascending=[False, True])
cv_summary

,rmse,mae,r2,pnl,zscore
OLS,0.000712,0.000509,0.962375,4.161082,0.758836
BayesianRidge,0.000712,0.000509,0.962367,4.161082,0.758836
Ridge,0.000712,0.000509,0.962363,4.161082,0.758836
ElasticNet,0.001007,0.000630,0.932352,4.160946,0.758801
Lasso,0.000749,0.000514,0.959814,4.157702,0.758040
Huber,0.000726,0.000507,0.961910,4.160558,0.758024
GBRT,0.001123,0.000413,0.928888,4.156190,0.757510
RandomForest,0.001155,0.000417,0.924853,4.151337,0.756877
ExtraTrees,0.001102,0.000391,0.931620,4.151618,0.756382
CART,0.001570,0.000578,0.859502,4.142294,0.750678


In [15]:
# Out-of-sample (held-out test): refit on full train+val, evaluate on test
oos_rows = []
fitted_regs = {}
for name, model in reg_models.items():
    model.fit(X_tv, y_ret_tv)
    fitted_regs[name] = model
    pred = model.predict(X_test)
    # actual pnl from trading the prediction
    pnls = np.sign(pred) * y_ret_test
    eq = (1.0 + pnls).cumprod()
    std = pnls.std()
    z = np.mean(pnls) / std
    oos_rows.append(
        {
            "model": name,
            "rmse": np.sqrt(mean_squared_error(y_ret_test, pred)),
            "mae": mean_absolute_error(y_ret_test, pred),
            "r2": r2_score(y_ret_test, pred),
            "pnl": eq.iloc[-1],
            "zscore": z,
        }
    )

oos_regression = pd.DataFrame(oos_rows).set_index("model")
# sort models by zscore then rmse
oos_regression = oos_regression.sort_values(by=["zscore", "rmse"], ascending=[False, True])
oos_regression

,rmse,mae,r2,pnl,zscore
model,,,,,
ExtraTrees,0.000411,0.000305,0.984867,4.350092,0.826839
Lasso,0.000655,0.000502,0.961629,4.340661,0.824791
RandomForest,0.000415,0.000308,0.984586,4.338723,0.824370
Ridge,0.000650,0.000516,0.962160,4.336271,0.823838
BayesianRidge,0.000650,0.000516,0.962158,4.336271,0.823838
OLS,0.000650,0.000516,0.962156,4.336271,0.823838
Huber,0.000650,0.000502,0.962146,4.333741,0.823289
GBRT,0.000437,0.000333,0.982900,4.333036,0.823136
CART,0.000543,0.000357,0.973568,4.303337,0.816703


### Classification Models

In [16]:
# Classification models (LR + Trees)
cls_models = {
    "LogisticRegression": Pipeline([("scaler", StandardScaler()), ("est", LogisticRegression(max_iter=5_000, random_state=42))]),
    "CART": DecisionTreeClassifier(max_depth=10, random_state=42, min_samples_leaf=15),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(random_state=42, max_depth=4, n_estimators=200, learning_rate=0.05),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_depth=4, max_iter=200, learning_rate=0.05, random_state=42
    ),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
        n_estimators=150,
        learning_rate=0.5,
        random_state=42,
    ),
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=8, random_state=42),
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    ),
}


In [17]:

cv_acc = []
for name, model in cls_models.items():
    accs = []
    for tr, va in tscv.split(X_tv):
        m = clone(model)
        m.fit(X_tv.iloc[tr], y_dir_tv.iloc[tr])
        pred = m.predict(X_tv.iloc[va])
        accs.append(accuracy_score(y_dir_tv.iloc[va], pred))
    cv_acc.append({"model": name, "cv_accuracy_mean": float(np.mean(accs)), "cv_accuracy_std": float(np.std(accs))})

fitted_cls = {}
for name, model in cls_models.items():
    m = clone(model)
    m.fit(X_tv, y_dir_tv)
    fitted_cls[name] = m

pd.DataFrame(cv_acc)

,model,cv_accuracy_mean,cv_accuracy_std
0,LogisticRegression,0.907491,0.024762
1,CART,0.887184,0.019336
2,ExtraTrees,0.904332,0.020778
3,RandomForest,0.898917,0.029660
4,GradientBoosting,0.897563,0.015547
5,HistGradientBoosting,0.894404,0.023483
6,AdaBoost,0.886282,0.015263
7,Bagging,0.898466,0.020635


In [18]:
# Confusion matrices on held-out test (one per classifier)
# sklearn: rows = true class, columns = predicted class; C[i,j] = # with y_true=i and y_pred=j
print(
    """
3×3 confusion_matrix (labels=[0,1,2]):
              predicted class
           j=0        j=1        j=2
  i=0   [ C[0,0]    C[0,1]     C[0,2] ]   true 0 → pred 0 / 1 / 2
  i=1   [ C[1,0]    C[1,1]     C[1,2] ]   true 1 → pred 0 / 1 / 2
  i=2   [ C[2,0]    C[2,1]     C[2,2] ]   true 2 → pred 0 / 1 / 2

  Diagonal C[i,i]: correct counts for class i.
  Off-diagonal C[i,j] (i≠j): misclassified as j when true class was i.
""".strip()
)
print()

3×3 confusion_matrix (labels=[0,1,2]):
              predicted class
           j=0        j=1        j=2
  i=0   [ C[0,0]    C[0,1]     C[0,2] ]   true 0 → pred 0 / 1 / 2
  i=1   [ C[1,0]    C[1,1]     C[1,2] ]   true 1 → pred 0 / 1 / 2
  i=2   [ C[2,0]    C[2,1]     C[2,2] ]   true 2 → pred 0 / 1 / 2

  Diagonal C[i,i]: correct counts for class i.
  Off-diagonal C[i,j] (i≠j): misclassified as j when true class was i.



In [19]:
nb_classes = np.unique(y_dir_test).size
print(f'The models fit {nb_classes} classes. Therefore, the confusion matrix is a {nb_classes}x{nb_classes} matrix.')
print(f'The class labels are {np.unique(y_dir_test)}')

The models fit 3 classes. Therefore, the confusion matrix is a 3x3 matrix.
The class labels are [-1.  0.  1.]


In [20]:
for name, model in fitted_cls.items():
    pred = model.predict(X_test)
    print(name, "\n", confusion_matrix(y_dir_test, pred))

LogisticRegression 
 [[303   0  23]
 [  0   4   1]
 [ 40   0 321]]
CART 
 [[297   0  29]
 [  1   0   4]
 [ 50   0 311]]
ExtraTrees 
 [[299   0  27]
 [  0   5   0]
 [ 43   0 318]]
RandomForest 
 [[305   0  21]
 [  2   1   2]
 [ 43   0 318]]
GradientBoosting 
 [[300   0  26]
 [  0   5   0]
 [ 52   0 309]]
HistGradientBoosting 
 [[299   0  27]
 [  1   3   1]
 [ 46   1 314]]
AdaBoost 
 [[307   0  19]
 [  0   4   1]
 [ 62   0 299]]
Bagging 
 [[302   0  24]
 [  2   2   1]
 [ 43   0 318]]


In [ ]:
# rank models by accuracy and report precision, recall
rows = []
for name, model in fitted_cls.items():
    pred = model.predict(X_test)
    precision = precision_score(y_dir_test, pred, average='macro')
    recall = recall_score(y_dir_test, pred, average='macro')
    rows.append({'model': name, 'precision': precision, 'recall': recall})
df_cls = pd.DataFrame(rows).set_index('model').sort_values(by='precision', ascending=False)
df_cls

/opt/miniconda3/envs/algotrading/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



,precision,recall
model,,
LogisticRegression,0.937939,0.872882
RandomForest,0.934660,0.672156
Bagging,0.932477,0.735756
ExtraTrees,0.932003,0.932688
GradientBoosting,0.924887,0.925400
AdaBoost,0.923094,0.856658
HistGradientBoosting,0.844097,0.795661
CART,0.585839,0.590846


In [21]:
# Test-period trading: classification (long if pred up, else short)
def equity_drawdown(pos: pd.Series, r: pd.Series):
    strat_r = pos * r
    eq = (1.0 + strat_r).cumprod()
    peak = eq.cummax()
    dd = eq / peak - 1.0
    return eq, dd

fig_cls = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Cumulative equity (test, start=1.0)", "Drawdown"),
    row_heights=[0.62, 0.38],
)

for name, model in fitted_cls.items():
    pred = model.predict(X_test)
    pos = pd.Series(np.where(pred == 1, 1.0, -1.0), index=X_test.index)
    eq, dd = equity_drawdown(pos, y_ret_test)
    fig_cls.add_trace(go.Scatter(x=eq.index, y=eq, name=name, mode="lines"), row=1, col=1)
    fig_cls.add_trace(go.Scatter(x=dd.index, y=dd, name=f"{name}_dd", line=dict(dash="dot"), showlegend=False), row=2, col=1)

fig_cls.update_layout(height=700, title="Classification models — test equity & drawdown", hovermode="x unified")
fig_cls.update_yaxes(title_text="equity", row=1, col=1)
fig_cls.update_yaxes(title_text="drawdown", row=2, col=1)
fig_cls.show()